# YOLO11l UCF-Crime2Local — full script pipeline

Runs the same steps as [README.md](README.md), in order, from the repository root.

**Before you run:** set `REPO_ROOT` and `UCFCRIME2LOCAL_ROOT` in the next cell. The dataset folder must contain `rgb-images/`, `labels/`, `train.txt`, `test.txt`, and `labels.txt`.

**GPU:** for CUDA (e.g. Google Colab), use `TRAIN_DEVICE = "0"`. On Apple Silicon use `"mps"`, CPU use `"cpu"`.

In [1]:
!git clone https://github.com/Shad0w-s/YoLoV11UCFCrime.git

Cloning into 'YoLoV11UCFCrime'...
remote: Enumerating objects: 63, done.
remote: Counting objects: 100% (63/63), done.
remote: Compressing objects: 100% (58/58), done.
remote: Total 63 (delta 5), reused 59 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (63/63), 4.57 MiB | 19.33 MiB/s, done.
Resolving deltas: 100% (5/5), done.


In [2]:
!curl -L -o ucfcrime2local.zip https://www.kaggle.com/api/v1/datasets/download/vulamnguyen/ucfcrime2local-with-ground-truth-bounding-boxes
!mkdir -p ucfcrime2local
!unzip -q ucfcrime2local.zip -d ucfcrime2local
!rm ucfcrime2local.zip

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 1223M  100 1223M    0     0  15.9M      0  0:01:16  0:01:16 --:--:-- 16.3M


In [11]:
import os
from pathlib import Path

# --- updated paths ---
REPO_ROOT = Path("/content/YoLoV11UCFCrime").resolve()
UCFCRIME2LOCAL_ROOT = "/content/ucfcrime2local/ucfcrime2local"

# Training device: "0" = first CUDA GPU, "mps", "cpu"
TRAIN_DEVICE = "0"

os.environ["UCFCRIME2LOCAL_ROOT"] = str(Path(UCFCRIME2LOCAL_ROOT).expanduser().resolve())
os.chdir(REPO_ROOT)
print("cwd:", os.getcwd())
print("UCFCRIME2LOCAL_ROOT:", os.environ["UCFCRIME2LOCAL_ROOT"])

cwd: /content/YoLoV11UCFCrime
UCFCRIME2LOCAL_ROOT: /content/ucfcrime2local/ucfcrime2local


## 0. Dependencies

Run once per environment (or skip if already installed).

In [7]:
!pip install -q -r requirements.txt

## 1. Inspect dataset

Writes `outputs/dataset_summary.md`.

In [12]:
!python scripts/inspect_dataset.py --out outputs/dataset_summary.md

# UCF-Crime2Local inspection
- **Dataset root:** `/content/ucfcrime2local/ucfcrime2local`
- **Inputs:** extracted frames under `rgb-images/<class>/<video>/<frame>.jpg`.
- **Labels:** YOLO txt per frame under `labels/...` (same relative path as list files).
- **Classes (6):** `0:Arrest`, `1:Assault`, `2:Burglary`, `3:Robbery`, `4:Stealing`, `5:Vandalism`
- **train.txt lines:** 31687 (unique videos: 65)
- **test.txt lines:** 13226 (unique videos: 31)
- **Train/test video overlap:** 0 (should be 0)

## Sample checks (first 20k train lines)

- Missing label files: 0
- Missing image files: 0
- Empty label files: 0
- Class id histogram (raw box rows): {0: 6541, 1: 3091, 2: 5109, 3: 5259}

Wrote outputs/dataset_summary.md


## 2. Split videos (train / val / test)

Writes `data/splits/*.txt`. Preserves official `test.txt`; train/val from `train.txt`.

In [13]:
!python scripts/split_videos.py --seed 42

Train videos: 55
Val videos:   10
Test videos:  31
Wrote lists under /content/YoLoV11UCFCrime/data/splits


In [14]:
!find /content/ucfcrime2local -maxdepth 2

/content/ucfcrime2local
/content/ucfcrime2local/ucfcrime2local
/content/ucfcrime2local/ucfcrime2local/rgb-images
/content/ucfcrime2local/ucfcrime2local/labels.txt
/content/ucfcrime2local/ucfcrime2local/train.txt
/content/ucfcrime2local/ucfcrime2local/labels
/content/ucfcrime2local/ucfcrime2local/test.txt


## 3. Convert to YOLO layout

Populates `data/processed/` and `data/processed/data.yaml`.

In [15]:
!python scripts/convert_to_yolo.py

Processed frames: {'train': 25769, 'val': 5918, 'test': 13226}
Wrote /content/YoLoV11UCFCrime/data/processed/data.yaml


## 4. Validate dataset + debug previews

Writes sample overlays to `outputs/debug_samples/`.

In [16]:
!python scripts/validate_yolo_dataset.py --num-samples 20

Train images scanned: 25769
Missing label files: 0
Invalid label files (counted): 0
Wrote 20 previews to /content/YoLoV11UCFCrime/outputs/debug_samples


## 5. Train YOLO11l

Default: 100 epochs, batch 16, outputs under `runs_ucfcrime/yolo11l_anomaly_region/` (or under `runs/detect/...` depending on Ultralytics layout).

If you run out of memory, lower `--batch` or use `--cache false` or `--cache disk`.

In [ ]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "scripts/train_yolo11l.py", "--batch", "16", "--device", TRAIN_DEVICE],
    check=True,
)

## 6. Inference on test set

Uses trained `best.pt` if present (script picks default paths).

In [ ]:
!python scripts/infer_yolo.py --weights runs_ucfcrime/yolo11l_anomaly_region/weights/best.pt

## 7. Frame anomaly scores

Writes `outputs/predictions/frame_scores_yolo.csv` (and related outputs).

In [ ]:
!python scripts/build_frame_scores.py --smooth-window 5

## 8. Detection metrics (mAP, etc.)

Runs `model.val()` and writes `outputs/metrics/metrics_detection.json`.

In [ ]:
!python scripts/evaluate_detection.py --weights runs_ucfcrime/yolo11l_anomaly_region/weights/best.pt

## 9. Anomaly metrics (optional — needs frame-level labels CSV)

Without `--frame-labels`, the script skips ROC-AUC/AP and only reminds you that scores exist.

With labels: CSV columns `frame_path` + `label`, or `video_id` + `frame_id` + `label`.

In [ ]:
# No labels (skip metrics file):
!python scripts/evaluate_anomaly.py

# Uncomment and set path when you have a labels CSV:
# !python scripts/evaluate_anomaly.py --frame-labels /path/to/frame_labels.csv

## 10. Visualization examples (optional)

In [ ]:
!python scripts/visualize_examples.py --weights runs_ucfcrime/yolo11l_anomaly_region/weights/best.pt

## 11. Merge YOLO + VadCLIP metrics (optional)

Requires `outputs/metrics/metrics_anomaly.json` from step 9 **with** frame labels. Uncomment when ready.

In [ ]:
# !python scripts/merge_metrics_optional.py --vadclip /path/to/vadclip_metrics.json